<a href="https://colab.research.google.com/github/rdelhibabu/ML-Entanglement-Witness-2026/blob/main/ML_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

def generate_quantum_dataset(num_samples=10000, gamma=1.0):
    """
    Generates the quantum dataset based on the correlated dephasing channel
    applied to the maximally entangled Bell state |Phi+>.
    """
    # Pauli matrices for measurement extraction
    I = np.array([[1, 0], [0, 1]])
    X = np.array([[0, 1], [1, 0]])
    Y = np.array([[0, -1j], [1j, 0]])
    Z = np.array([[1, 0], [0, -1]])
    paulis = [I, X, Y, Z]

    data = []

    # Randomly sample time t and correlation parameter xi
    # t in [0, 5/gamma], xi in [0, 1]
    np.random.seed(42) # For reproducibility
    times = np.random.uniform(0, 5/gamma, num_samples)
    xis = np.random.uniform(0, 1.0, num_samples)

    for t, xi in zip(times, xis):
        # Construct the analytical time-evolved density matrix (Eq. 2 & 5)
        rho = np.zeros((4, 4), dtype=complex)

        # Diagonal classical populations
        rho[0, 0] = 0.5
        rho[3, 3] = 0.5

        # Off-diagonal coherences protected by the correlated channel (Eq 5)
        coherence = 0.5 * ((1 - xi) * np.exp(-2 * gamma * t) + xi)
        rho[0, 3] = coherence
        rho[3, 0] = coherence

        # Extract 9-dimensional Pauli Features (c_ij)
        # We need c11 to c33 (skipping index 0 which is Identity)
        features = {}
        for i in range(1, 4):
            for j in range(1, 4):
                observable = np.kron(paulis[i], paulis[j])
                # Expectation value Tr(rho * (sigma_i x sigma_j))
                val = np.real(np.trace(rho @ observable))
                features[f'c{i}{j}'] = val

        # Evaluate true separability via Partial Transpose (PPT Criterion)
        # Swap the inner subsystem indices for a 2x2 system
        rho_pt = np.zeros((4, 4), dtype=complex)
        for i in range(2):
            for j in range(2):
                for k in range(2):
                    for l in range(2):
                        rho_pt[2*i+k, 2*j+l] = rho[2*i+l, 2*j+k]

        eigenvalues = np.linalg.eigvals(rho_pt)
        min_eig = np.min(np.real(eigenvalues))

        # Label: 1 if physically entangled (negative eigenvalue), 0 if separable
        is_entangled = 1 if min_eig < -1e-10 else 0

        # Standard Linear Witness check: W = 0.5*I - |Phi+><Phi+|
        witness_val = -0.5 * ((1 - xi) * np.exp(-2 * gamma * t) + xi)
        witness_detected = 1 if witness_val < -1e-10 else 0

        # Record the row
        row = {
            'time': t,
            'xi': xi,
            **features,
            'min_eig_PPT': min_eig,
            'label_entangled': is_entangled,
            'witness_val': witness_val,
            'witness_detected': witness_detected
        }
        data.append(row)

    df = pd.DataFrame(data)
    return df

# Generate and save the dataset
print("Generating dataset...")
quantum_df = generate_quantum_dataset(num_samples=100000)

# Save to CSV
csv_filename = 'quantum_entanglement_dataset.csv'
quantum_df.to_csv(csv_filename, index=False)
print(f"Dataset successfully generated and saved as '{csv_filename}'.")
print(quantum_df.head())

Generating dataset...
Dataset successfully generated and saved as 'quantum_entanglement_dataset.csv'.
       time        xi       c11  c12  c13  c21       c22  c23  c31  c32  c33  \
0  1.872701  0.580779  0.590684  0.0  0.0  0.0 -0.590684  0.0  0.0  0.0  1.0   
1  4.753572  0.526972  0.527007  0.0  0.0  0.0 -0.527007  0.0  0.0  0.0  1.0   
2  3.659970  0.351037  0.351467  0.0  0.0  0.0 -0.351467  0.0  0.0  0.0  1.0   
3  2.993292  0.493213  0.494486  0.0  0.0  0.0 -0.494486  0.0  0.0  0.0  1.0   
4  0.780093  0.365097  0.498488  0.0  0.0  0.0 -0.498488  0.0  0.0  0.0  1.0   

   min_eig_PPT  label_entangled  witness_val  witness_detected  
0    -0.295342                1    -0.295342                 1  
1    -0.263503                1    -0.263503                 1  
2    -0.175733                1    -0.175733                 1  
3    -0.247243                1    -0.247243                 1  
4    -0.249244                1    -0.249244                 1  
